In [ ]:
import os
import load_dotenv
from load_dotenv import load_dotenv

# This function will load all the variable from the .env file and 
# make them available in the os.environ dictionary (env variables)
load_dotenv()

if os.environ.get("CLAUDE_API_KEY"):
    print("API KEY variable has been loaded")
else:
    raise ValueError("CLAUDE API KEY not found")

if os.environ.get("TAVILY_API_KEY"):
    print("TAVILY API KEY variable has been loaded")
else:
    raise ValueError("TAVILY API KEY not found")

In [ ]:
from langchain_anthropic import ChatAnthropic

llm_anthropic = ChatAnthropic(
    model=os.environ.get("CLAUDE_MODEL"), 
    temperature=0,
    api_key=os.environ.get("CLAUDE_API_KEY"),
    base_url=os.environ.get("CLAUDE_BASE_URL")
)
llm_anthropic

#### **TOOLS**

In [7]:
# TOOL - 1 [News Search Tool]

from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun(description= "This is a tool to search the news in the web")

search_tool.invoke("Who is Sayani Dutta ?")

# # TOOL - 2 [Wikipedia Search Tool]

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia_tool = WikipediaQueryRun(api_wrapper = WikipediaAPIWrapper(), description="This is a Wikipedia tool to search Wikipedia")

wikipedia_tool.invoke("What is the capital of India ?")

'Page: Delhi\nSummary: Delhi, officially the National Capital Territory (NCT) of Delhi, is a megacity and a union territory of India containing New Delhi, the capital of India. Straddling the Yamuna river, but spread chiefly to the west, or beyond its right bank, Delhi shares borders with the state of Uttar Pradesh in the east and with the state of Haryana in the remaining directions. Delhi became a union territory on 1 November 1956 and the NCT in 1995. The NCT covers an area of 1,484 square kilometres (573 sq mi). According to the 2011 census, Delhi\'s city proper population was over 11 million, while the NCT\'s population was about 16.8 million.\nThe topography of the medieval fort Purana Qila on the banks of the river Yamuna matches the literary description of the citadel Indraprastha in the Sanskrit epic Mahabharata; however, excavations in the area have revealed no signs of an ancient built environment.\nFrom the early 13th century until the mid-19th century, Delhi was the capita

In [ ]:
# TOOL - 1 [Tavily Search Tool]
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    api_key = os.environ.get('TAVILY_API_KEY'),
    max_results = 5,
    description = "This is a Tavily tool to search the web for current information"
)

search_tool.invoke("Who is Sayani Dutta ?")

In [ ]:
# TOOL - 3 [Custom Enterprise Tool]
from langchain.tools import tool 

@tool
def enterprise_tool(query:str)-> str:

    """This custom tool sends an email to the employees"""
    return "Email sent"

In [ ]:
Toolkit = [search_tool, wikipedia_tool, enterprise_tool]
Toolkit

#### ReAct Agent

In [ ]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model = os.environ.get("CLAUDE_MODEL"), 
    temperature = 0.1,
    api_key = os.environ.get("CLAUDE_API_KEY"),
    base_url = os.environ.get("CLAUDE_BASE_URL"),
    max_tokens = 1000,
    timeout = 30
)

agent = create_agent(model, tools = Toolkit)
agent

#### ReAct Agent Invoke with Streams

In [ ]:
example_query = "Give me the latest news of StockMarket"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode = "values",
)

for event in events:
    event["messages"][-1].pretty_print()

#### Manually Binding the LLM with Tools

In [ ]:
# Without Binding

result = llm_anthropic.invoke("What is the latest trending news about stock market today ?")
result.content

In [ ]:
# With Binding

llm_binded = llm_anthropic.bind_tools(Toolkit)
new_result = llm_binded.invoke("What is the latest trending news about stock market today ?")
new_result.pretty_print()